# Inference using LSDB-style API

In [1]:
import lsdb
import hyrax

In [2]:
from distributed import Client, LocalCluster

N_WORKERS, THREADS_PER_WORKER = 1, 2
cluster = LocalCluster(n_workers=N_WORKERS, threads_per_worker=THREADS_PER_WORKER, processes=True)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 1
Total threads: 2,Total memory: 30.89 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34583,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34579,Total threads: 2
Dashboard: http://127.0.0.1:37179/status,Memory: 30.89 GiB
Nanny: tcp://127.0.0.1:44583,


## Open a HATS catalog with LSDB

Open a TESS catalog with LSDB and use LSDB's `map_partitions` to apply a function to each row in the catalog. In this example, we're applying a function that removes nested rows that contain NaNs.

In [3]:
from lsdb import ConeSearch

tess = lsdb.open_catalog(
    "https://data.lsdb.io/hats/tess/tess_lightcurve",
    columns=["ticid", "ra_obj", "dec_obj", "lightcurve"],
)

cone = ConeSearch(ra=180.0, dec=0.0, radius_arcsec=1 * 3600)
tess_cone = tess.search(cone)


def drop_nans(df):
    return df.dropna(subset="lightcurve.sap_flux").dropna(subset="lightcurve")


tess_filtered_nans = tess_cone.map_partitions(drop_nans)

In [4]:
h = hyrax.Hyrax()
h.config["model"]["name"] = "HyraxLoopback"
h.config["data_loader"]["batch_size"] = 64

h.config["data_request"] = {
    "infer": {
        "data": {
            "dataset_class": "DataframeDataset",
            "data_location": [],  # placeholder
            "primary_id_field": "ticid",
            "fields": ["ra_obj", "dec_obj", "lightcurve_sap_flux"],
        }
    }
}

h.set_config("infer.model_weights_file", "none")

## Use a Dask Worker Plugin to provide a model for use with inference

In [5]:
import uuid

import torch
import numpy as np
import pandas as pd
from distributed import WorkerPlugin, get_worker

from hyrax.models.model_registry import fetch_model_class
from hyrax.models.model_utils import load_model_weights
from hyrax.pytorch_ignite import _auto_model, create_process_func, dist_data_loader, setup_dataset

PRED_META = pd.DataFrame(
    {
        "pred": pd.Series(dtype="float32"),
    }
)


def predict(df, process_fn, dataset, config):
    with torch.no_grad():
        dataset.data = df
        data_loader = dist_data_loader(dataset, config)

        for b in data_loader:
            model_out = process_fn(b)
            print(model_out)
            z = model_out["ra_obj"] if len(b) else np.zeros(0, np.float32)
    return pd.DataFrame({"pred": z}, index=df.index)


class TorchModelPlugin(WorkerPlugin):
    def __init__(self, name, config):
        self.name = name  # plugins are keyed by name on the worker
        self.model = None
        self.config = config

    def setup(self, worker):
        model_cls = fetch_model_class(self.config)
        model = model_cls(self.config, None)
        # load_model_weights(self.config, model, 'infer')
        #! Should go ahead and load the model prepare_inputs function and attach it to the model
        # from hyrax.plugin_utils import load_prepare_inputs
        # model.prepare_inputs = load_prepare_inputs(<input_directory>)
        model.load_token = uuid.uuid4().int >> 65
        model.eval()
        wrapped_model = _auto_model(model)
        self.model = wrapped_model
        self.process_fn = create_process_func("infer_batch", "cuda", self.model, self.config)

        # setup the dataset class
        print(self.config["data_request"])
        datasets = setup_dataset(self.config, shuffle=False)
        self.dataset = datasets["infer"]

    def teardown(self, worker):
        self.model = None


def predict_with_plugin(df, plugin_name):
    return predict(
        df,
        get_worker().plugins[plugin_name].process_fn,
        get_worker().plugins[plugin_name].dataset,
        get_worker().plugins[plugin_name].config,
    )

In [6]:
client.register_plugin(TorchModelPlugin("test-model", h.config))
out = tess_filtered_nans.map_partitions(predict_with_plugin, plugin_name="test-model", meta=PRED_META)

out.compute()

{'infer': {'data': {'dataset_class': 'DataframeDataset', 'data_location': [], 'primary_id_field': 'ticid', 'fields': ['ra_obj', 'dec_obj', 'lightcurve_sap_flux']}}}


No `get_ra_obj` method for requested field, 'ra_obj' was found in dataset DataframeDataset.
No `get_dec_obj` method for requested field, 'dec_obj' was found in dataset DataframeDataset.
No `get_lightcurve_sap_flux` method for requested field, 'lightcurve_sap_flux' was found in dataset DataframeDataset.
No `get_*` methods were found in the class: DataframeDataset. This is likely an error in the dataset class definition.


Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

2026-09-28 16:57:40,496 ignite.distributed.auto.auto_dataloader INFO: Use data loader kwargs for dataset '<torch.utils.data.da': 
	{'sampler': None, 'batch_size': 64, 'collate_fn': <bound method CollationMixin.collate of Name: data (primary dataset)
  Dataset class: DataframeDataset
  Data location: []
  Primary ID field: ticid
  Requested fields: ra_obj, dec_obj, lightcurve_sap_flux
>, 'pin_memory': True}
2026-09-28 16:57:40,595 - distributed.worker - ERROR - Compute Failed
Key:       ('predict_with_plugin-3779843ed643677824f50990c45d9681', 2)
State:     executing
Task:  <Task ('predict_with_plugin-3779843ed643677824f50990c45d9681', 2) wrapped_func(..., ...)>
Exception: 'RuntimeError("Error applying function predict_with_plugin to partition 2: cannot access local variable \'z\' where it is not associated with a value")'
Traceback: '  File "/home/drew/miniconda3/envs/hyrax/lib/python3.13/site-packages/lsdb/operations/lsdb_ops.py", line 38, in wrapper\n    return run_and_verify_meta(fun

RuntimeError: Error applying function predict_with_plugin to partition 2: cannot access local variable 'z' where it is not associated with a value

2026-09-28 16:57:43,468 ignite.distributed.auto.auto_dataloader INFO: Use data loader kwargs for dataset '<torch.utils.data.da': 
	{'sampler': None, 'batch_size': 64, 'collate_fn': <bound method CollationMixin.collate of Name: data (primary dataset)
  Dataset class: DataframeDataset
  Data location: []
  Primary ID field: ticid
  Requested fields: ra_obj, dec_obj, lightcurve_sap_flux
>, 'pin_memory': True}
2026-09-28 16:57:43,471 - distributed.worker - ERROR - Compute Failed
Key:       ('predict_with_plugin-3779843ed643677824f50990c45d9681', 1)
State:     executing
Task:  <Task ('predict_with_plugin-3779843ed643677824f50990c45d9681', 1) wrapped_func(..., ...)>
Exception: 'RuntimeError("Error applying function predict_with_plugin to partition 1: cannot access local variable \'z\' where it is not associated with a value")'
Traceback: '  File "/home/drew/miniconda3/envs/hyrax/lib/python3.13/site-packages/lsdb/operations/lsdb_ops.py", line 38, in wrapper\n    return run_and_verify_meta(fun

In [ ]:
client.close()
cluster.close()